# 🧩 Exercícios — Busca em Largura (BFS)

Este notebook implementa soluções para os exercícios propostos no material de BFS.

Exercícios abordados:
1. Retornar níveis (camadas) como listas.
2. Floresta BFS para grafos desconexos.
3. Teste de bipartição (2-coloração) com BFS.
4. Alcance em digrafos a partir de uma fonte.
5. Comparação de distâncias da BFS com NetworkX.

## Importações

In [ ]:
from collections import deque
import networkx as nx
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Any, Optional

## 1) BFS retornando níveis (camadas)

In [ ]:
def bfs_camadas(G: nx.Graph, s) -> Tuple[List[List[Any]], Dict[Any, Optional[int]], Dict[Any, Optional[Any]]]:
    """
    Executa BFS a partir de s e retorna:
      - levels: lista de listas, onde levels[k] contém os vértices a distância k.
      - d: distâncias (em número de arestas) de s até v.
      - pai: predecessor de cada v na árvore BFS.
    Vértices inalcançáveis não aparecem em nenhuma camada.
    """
    if s not in G:
        raise ValueError('Fonte s não pertence ao grafo')
    cor = {v: 'white' for v in G.nodes()}
    d   = {v: None for v in G.nodes()}
    pai = {v: None for v in G.nodes()}
    levels: List[List[Any]] = []

    Q = deque()
    cor[s] = 'gray'; d[s] = 0; Q.append(s)
    levels.append([s])

    while Q:
        u = Q.popleft()
        for v in G.neighbors(u):
            if cor[v] == 'white':
                cor[v] = 'gray'
                d[v] = d[u] + 1
                pai[v] = u
                if len(levels) <= d[v]:
                    levels.append([])
                levels[d[v]].append(v)
                Q.append(v)
        cor[u] = 'black'
    return levels, d, pai

In [ ]:
# Demonstração rápida
G = nx.Graph([(1,2),(1,3),(2,4),(2,5),(3,6),(5,6),(6,7)])
levels, d, pai = bfs_camadas(G, 1)
print('Levels:', levels)
print('d:', d)
print('pai:', pai)

## 2) Floresta BFS (grafos desconexos)
Para grafos desconexos, executamos BFS a partir de cada vértice ainda não visitado, produzindo uma **floresta** de árvores BFS (uma por componente).

In [ ]:
def floresta_bfs(G: nx.Graph):
    """
    Retorna uma lista de componentes, onde cada componente é um dicionário com:
      { 'raiz': r, 'arvore_arestas': list[(u,v)], 'd': {...}, 'pai': {...}, 'vertices': set(...) }
    As arestas são orientadas da raiz para os filhos para facilitar visualização.
    """
    visitado = {v: False for v in G.nodes()}
    componentes = []

    for s in G.nodes():
        if not visitado[s]:
            # BFS nesta componente
            cor = {v: 'white' for v in G.nodes()}
            d   = {v: None for v in G.nodes()}
            pai = {v: None for v in G.nodes()}
            arestas = []
            Q = deque([s])
            cor[s] = 'gray'; d[s] = 0
            comp_vertices = set([s])
            visitado[s] = True
            while Q:
                u = Q.popleft()
                for v in G.neighbors(u):
                    if cor[v] == 'white':
                        cor[v] = 'gray'
                        d[v] = d[u] + 1
                        pai[v] = u
                        arestas.append((u, v))
                        Q.append(v)
                        visitado[v] = True
                        comp_vertices.add(v)
                cor[u] = 'black'

            componentes.append({
                'raiz': s,
                'arvore_arestas': arestas,
                'd': d,
                'pai': pai,
                'vertices': comp_vertices,
            })
    return componentes

In [ ]:
# Demonstração com duas componentes
H = nx.Graph()
H.add_edges_from([(1,2),(2,3),(4,5)])
comps = floresta_bfs(H)
for i, comp in enumerate(comps, 1):
    print(f'Componente {i}: raiz={comp[
]}, vertices={sorted(comp[
])}, arestas={comp[
]}')

## 3) Bipartição via BFS (2-coloração)
Um grafo não dirigido é bipartido se e somente se não contém ciclos de comprimento ímpar. Podemos usar BFS para 2-colorir os vértices: ao visitar um vizinho, atribuímos a cor oposta. Se encontrarmos uma aresta entre vértices de mesma cor, o grafo não é bipartido.

In [ ]:
def bipartido_bfs(G: nx.Graph):
    """
    Retorna (is_bipartite, color) onde color[v] ∈ {0,1} quando bipartido.
    Se não bipartido, retorna (False, color_parcial).
    """
    color: Dict[Any, Optional[int]] = {v: None for v in G.nodes()}
    for s in G.nodes():
        if color[s] is None:
            color[s] = 0
            Q = deque([s])
            while Q:
                u = Q.popleft()
                for v in G.neighbors(u):
                    if color[v] is None:
                        color[v] = 1 - color[u]
                        Q.append(v)
                    elif color[v] == color[u]:
                        return False, color
    return True, color

In [ ]:
# Exemplo bipartido
B = nx.complete_bipartite_graph(3, 2)
ok, color = bipartido_bfs(B)
print('Bipartido?', ok, '| color:', color)

# Exemplo não bipartido (triângulo)
T = nx.cycle_graph(3)
ok2, color2 = bipartido_bfs(T)
print('Bipartido?', ok2, '| color:', color2)

## 4) Alcance em digrafos com BFS
Dado um vértice origem s em um digrafo, computamos o conjunto de vértices alcançáveis seguindo a orientação das arestas.

In [ ]:
def alcancaveis_digrafo_bfs(D: nx.DiGraph, s):
    if s not in D:
        raise ValueError('Fonte s não pertence ao digrafo')
    visitado = set([s])
    Q = deque([s])
    ordem = []
    while Q:
        u = Q.popleft()
        ordem.append(u)
        for v in D.successors(u):
            if v not in visitado:
                visitado.add(v)
                Q.append(v)
    return visitado, ordem

In [ ]:
# Demonstração
D = nx.DiGraph()
D.add_edges_from([('A','B'),('B','C'),('C','A'),('C','D')])
reach, ordem = alcancaveis_digrafo_bfs(D, 'A')
print('Alcançáveis a partir de A:', reach)
print('Ordem de visita:', ordem)

## 5) Comparar distâncias com NetworkX
Validamos as distâncias calculadas pela BFS com os comprimentos de caminhos mínimos em grafos não ponderados do NetworkX.

In [ ]:
def comparar_bfs_com_networkx(G: nx.Graph, s) -> bool:
    # BFS própria
    cor = {v: 'white' for v in G.nodes()}
    d   = {v: None for v in G.nodes()}
    pai = {v: None for v in G.nodes()}
    from collections import deque
    Q = deque()
    cor[s] = 'gray'; d[s] = 0; Q.append(s)
    while Q:
        u = Q.popleft()
        for v in G.neighbors(u):
            if cor[v] == 'white':
                cor[v] = 'gray'; d[v] = d[u] + 1; pai[v] = u; Q.append(v)
        cor[u] = 'black'
    # Distâncias do NetworkX (apenas nós alcançáveis)
    try:
        nx_dists = nx.shortest_path_length(G, source=s)
    except nx.NetworkXNoPath:
        nx_dists = {}
    ok = True
    for v, dist in nx_dists:
        if d[v] != dist:
            print(f'Divergência em {v}: BFS={d[v]} vs nx={dist}')
            ok = False
    return ok

In [ ]:
# Experimento simples
import random
random.seed(42)
for n, m in [(20, 30), (30, 50), (50, 100)]:
    G = nx.gnm_random_graph(n, m, seed=42)
    s = 0
    ok = comparar_bfs_com_networkx(G, s)
    print(f'n={n}, m={m} -> distâncias iguais? {ok}')